In [ ]:
# Mount Google Drive and clone the repo
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/20020109tharindu/ML_Final_Project.git
%cd ML_Final_Project
!pip install imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

In [ ]:
# Load shared preprocessed splits from Google Drive
DATA_DIR = '/content/drive/MyDrive/FraudDetection/cleaned_data'

X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv').squeeze()
y_test  = pd.read_csv(f'{DATA_DIR}/y_test.csv').squeeze()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Fraud in train: {y_train.sum()} | Fraud in test: {y_test.sum()}')

In [ ]:
X_train.head()

In [ ]:
# Train logistic regression — lbfgs works well here, max_iter bumped to avoid convergence warnings
model = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0, random_state=42)
model.fit(X_train, y_train)
print('Training done.')

In [ ]:
# Predictions and probability scores
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall   :', round(recall_score(y_test, y_pred), 4))
print('F1-Score :', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.title('Confusion Matrix — Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score   = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall curve (more informative for imbalanced data)
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(rec_vals, prec_vals, color='darkorange', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients — shows which features push the model toward fraud
coef = pd.Series(model.coef_[0], index=X_train.columns)
top  = pd.concat([coef.nsmallest(10), coef.nlargest(10)])

plt.figure(figsize=(9, 7))
top.plot(kind='barh', color=['steelblue' if v < 0 else 'crimson' for v in top.values])
plt.axvline(0, color='black', lw=0.8, linestyle='--')
plt.title('Feature Coefficients — Logistic Regression')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# Threshold tuning — find the threshold that gives the best F1 on the test set
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall':    recall_score(y_test, preds, zero_division=0),
        'f1':        f1_score(y_test, preds, zero_division=0),
    })

tdf       = pd.DataFrame(results)
best_t    = tdf.loc[tdf['f1'].idxmax(), 'threshold']
best_row  = tdf.loc[tdf['f1'].idxmax()]

print(f'Best threshold: {best_t:.2f}')
print(f'  Precision: {best_row["precision"]:.4f}')
print(f'  Recall   : {best_row["recall"]:.4f}')
print(f'  F1-Score : {best_row["f1"]:.4f}')

In [ ]:
# Plot precision / recall / f1 vs threshold
plt.figure(figsize=(8, 5))
plt.plot(tdf['threshold'], tdf['precision'], label='Precision', color='steelblue')
plt.plot(tdf['threshold'], tdf['recall'],    label='Recall',    color='darkorange')
plt.plot(tdf['threshold'], tdf['f1'],        label='F1',        color='green')
plt.axvline(best_t, color='red', linestyle='--', label=f'Best ({best_t:.2f})')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final results at best threshold
y_pred_tuned = (y_proba >= best_t).astype(int)

print('=== Final Results (Logistic Regression) ===')
print(f'Threshold : {best_t:.2f}')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_tuned):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_tuned):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_tuned):.4f}')
print(f'ROC-AUC   : {auc_score:.4f}')